# 01 — K-Fold Split + YOLO Label Tree

Stratified 5-fold split over all 3,154 labeled images from `instances_train.json`.
Stratify on `has_annotations` so the 34 negatives spread across folds.

Outputs:
- `folds/fold_{0..4}_train.txt`, `fold_{0..4}_val.txt` — absolute image paths
- `folds/fold_{0..4}.yaml` — Ultralytics data configs
- `labels/train/{stem}.txt` — YOLO-format labels (one tree, shared across folds)
- `image_id_map.json` — `{filename_stem: coco_image_id}` for later test-side rewrite (train side here for reference)


In [1]:
import random, numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [2]:
import json, os
from pathlib import Path
from collections import defaultdict

PROJECT_ROOT = Path(r"C:/Users/Victor/Desktop/Projects/ESA_SAR_WBF")
SRC_ROOT    = Path(r"C:/Users/Victor/Desktop/Projects/ESA_SAR/ClearSAR")

TRAIN_IMG_DIR = SRC_ROOT / "data/images/train"
TEST_IMG_DIR  = SRC_ROOT / "data/images/test"
ANN_FILE      = SRC_ROOT / "data/annotations/instances_train.json"

FOLDS_DIR  = PROJECT_ROOT / "folds"
LABELS_DIR = PROJECT_ROOT / "labels/train"
FOLDS_DIR.mkdir(parents=True, exist_ok=True)
LABELS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAIN_IMG_DIR exists:", TRAIN_IMG_DIR.is_dir())
print("ANN_FILE exists:", ANN_FILE.is_file())

PROJECT_ROOT: C:\Users\Victor\Desktop\Projects\ESA_SAR_WBF
TRAIN_IMG_DIR exists: True
ANN_FILE exists: True


In [3]:
with open(ANN_FILE) as f:
    coco = json.load(f)

images = coco["images"]             # [{id, width, height, file_name}, ...]
anns   = coco["annotations"]         # [{image_id, bbox:[x,y,w,h], category_id, ...}]
cats   = coco["categories"]

ann_by_img = defaultdict(list)
for a in anns:
    ann_by_img[a["image_id"]].append(a)

n_total = len(images)
n_with  = sum(1 for im in images if ann_by_img[im["id"]])
n_empty = n_total - n_with
print(f"images={n_total}  with_anns={n_with}  empty={n_empty}  anns={len(anns)}  cats={[c['name'] for c in cats]}")
assert n_total == 3154, n_total
assert n_empty == 34, n_empty

images=3154  with_anns=3120  empty=34  anns=9288  cats=['RFI']


## Build YOLO label tree

One line per annotation: `class cx cy w h` (all normalized to image size). `class=0` since the single RFI class maps to index 0 in the Ultralytics YAML (even though COCO `category_id=1`).

In [4]:
written = 0
empty_written = 0
for im in images:
    stem = Path(im["file_name"]).stem
    W, H = im["width"], im["height"]
    lines = []
    for a in ann_by_img[im["id"]]:
        x, y, w, h = a["bbox"]
        cx = (x + w / 2) / W
        cy = (y + h / 2) / H
        nw = w / W
        nh = h / H
        lines.append(f"0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
    out = LABELS_DIR / f"{stem}.txt"
    out.write_text("\n".join(lines))
    written += 1
    if not lines:
        empty_written += 1
print(f"wrote {written} label files ({empty_written} empty for negative images)")

wrote 3154 label files (34 empty for negative images)


## Mirror image tree (symlink) so Ultralytics finds `images/train/*.png` next to `labels/train/*.txt`

Ultralytics resolves labels by replacing `/images/` with `/labels/` in the image path. To keep the source `data/images/train/` read-only (external), mirror the images under this project as a symlinked directory.

In [5]:
IMAGES_MIRROR = PROJECT_ROOT / "images/train"
(PROJECT_ROOT / "images").mkdir(exist_ok=True)
if not IMAGES_MIRROR.exists():
    # Directory junction on Windows — no admin required, unlike a true symlink.
    import subprocess
    cmd = f'mklink /J "{IMAGES_MIRROR}" "{TRAIN_IMG_DIR}"'
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout or r.stderr)
assert IMAGES_MIRROR.is_dir()
print("mirror:", IMAGES_MIRROR, "→", TRAIN_IMG_DIR)
sample = next(IMAGES_MIRROR.iterdir())
print("sample resolves:", sample)

Junction created for C:\Users\Victor\Desktop\Projects\ESA_SAR_WBF\images\train <<===>> C:\Users\Victor\Desktop\Projects\ESA_SAR\ClearSAR\data\images\train

mirror: C:\Users\Victor\Desktop\Projects\ESA_SAR_WBF\images\train → C:\Users\Victor\Desktop\Projects\ESA_SAR\ClearSAR\data\images\train
sample resolves: C:\Users\Victor\Desktop\Projects\ESA_SAR_WBF\images\train\1.png


## Stratified 5-fold split on `has_annotations`

In [6]:
from sklearn.model_selection import StratifiedKFold

img_ids  = [im["id"] for im in images]
stems    = [Path(im["file_name"]).stem for im in images]
y_strat  = [1 if ann_by_img[iid] else 0 for iid in img_ids]
paths    = [str((IMAGES_MIRROR / f"{s}.png").as_posix()) for s in stems]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_splits = list(skf.split(paths, y_strat))  # [(train_idx, val_idx), ...]

for i, (tr, va) in enumerate(fold_splits):
    tr_neg = sum(1 for k in tr if y_strat[k] == 0)
    va_neg = sum(1 for k in va if y_strat[k] == 0)
    print(f"fold {i}: train={len(tr)} (neg={tr_neg})  val={len(va)} (neg={va_neg})")

fold 0: train=2523 (neg=27)  val=631 (neg=7)
fold 1: train=2523 (neg=27)  val=631 (neg=7)
fold 2: train=2523 (neg=27)  val=631 (neg=7)
fold 3: train=2523 (neg=27)  val=631 (neg=7)
fold 4: train=2524 (neg=28)  val=630 (neg=6)


In [7]:
for i, (tr, va) in enumerate(fold_splits):
    tr_paths = [paths[k] for k in tr]
    va_paths = [paths[k] for k in va]
    (FOLDS_DIR / f"fold_{i}_train.txt").write_text("\n".join(tr_paths))
    (FOLDS_DIR / f"fold_{i}_val.txt").write_text("\n".join(va_paths))

    yaml_text = (
        f"path: {PROJECT_ROOT.as_posix()}\n"
        f"train: folds/fold_{i}_train.txt\n"
        f"val: folds/fold_{i}_val.txt\n"
        f"names:\n"
        f"  0: RFI\n"
    )
    (FOLDS_DIR / f"fold_{i}.yaml").write_text(yaml_text)
print("wrote folds/*.txt and folds/*.yaml")

wrote folds/*.txt and folds/*.yaml


## Sanity checks

- Every val image has a label file on disk.
- Folds are disjoint on val and union to the full set.
- Sample a random image+label pair and confirm YOLO coords are in [0,1].

In [8]:
all_val = set()
for i, (_, va) in enumerate(fold_splits):
    s = {paths[k] for k in va}
    assert not (all_val & s), f"fold {i} val overlaps earlier folds"
    all_val |= s
assert all_val == set(paths), "val union != all images"
print("fold val sets disjoint and cover all 3154 images ✓")

missing_labels = [s for s in stems if not (LABELS_DIR / f"{s}.txt").exists()]
print("missing label files:", len(missing_labels))
assert not missing_labels

# Coord range check on random sample
import random as _r
_r.seed(0)
for s in _r.sample(stems, 20):
    lf = (LABELS_DIR / f"{s}.txt").read_text().strip()
    if not lf:
        continue
    for line in lf.splitlines():
        _, cx, cy, w, h = line.split()
        for v in (cx, cy, w, h):
            assert 0.0 <= float(v) <= 1.0, (s, line)
print("YOLO label coords in [0,1] ✓")

fold val sets disjoint and cover all 3154 images ✓
missing label files: 0
YOLO label coords in [0,1] ✓


## Save filename-stem → image_id map

Train side is trivial (stems are numeric and already match IDs in the typical case), but stash the mapping anyway so 03_inference.ipynb has a template. Real work happens test-side once we have `instances_test.json` (or the challenge sample submission).

In [9]:
stem_to_id = {Path(im["file_name"]).stem: im["id"] for im in images}
mismatches = [(s, i) for s, i in stem_to_id.items() if not s.isdigit() or int(s) != i]
print(f"train stem==id mismatches: {len(mismatches)}")
(PROJECT_ROOT / "train_stem_to_id.json").write_text(json.dumps(stem_to_id))
print("wrote train_stem_to_id.json")

train stem==id mismatches: 0
wrote train_stem_to_id.json
